In [3]:
import sys

print(sys.executable)

/Users/tanya/VSCodeRandom/venv/bin/python


In [1]:
import pandas as pd
from scipy import stats
import math
import numpy as np

In [2]:
df = pd.read_csv("Salary.csv")
df_filtered = df.dropna()

# Task 1

According to open source data, the average salary in the country in 2022 was 64 200 руб. You have a sample for 2023. You predict that the average salary will decrease by 700. Test your hypothesis against the left-handed alternative with a significance level of 0.05

### First, let's inspect the data.

We assume that each employee appears only once in the dataset and that the sample was collected independently at random.

In [155]:
print("Describe salary data:")
df_filtered["Зарплата"].describe()

Describe salary data:


count      8122.000000
mean      62996.158828
std       25690.784074
min       12038.000000
25%       43301.000000
50%       61778.000000
75%       80983.250000
max      155021.000000
Name: Зарплата, dtype: float64

Let's inspect potential outliers.

In [ ]:
q1 = df_filtered["Зарплата"].quantile(0.25)  # 25th percentile (0.25 quantile)
q3 = df_filtered["Зарплата"].quantile(0.75)  # 75th percentile (0.75 quantile)

iqr = q3 - q1                       # interquartile range
lower = q1 - 1.5*iqr                # Whiskers: extend to the most extreme points
upper = q3 + 1.5*iqr

outliers = df_filtered[(df["Зарплата"] < lower) | (df_filtered["Зарплата"] > upper)]

print("Describe outliers:")
print("Lower part count:", outliers[outliers["Зарплата"] <= q1]["Зарплата"].count())
print("Upper part count:", outliers[outliers["Зарплата"] >= q3]["Зарплата"].count())



----- Describe outliers:

Lower part count: 0

Upper part count: 28


/var/folders/49/bfzf3lk97t9fwjpw00k_1j5w0000gp/T/ipykernel_3582/3985908856.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  outliers = df_filtered[(df["Зарплата"] < lower) | (df_filtered["Зарплата"] > upper)]


The outliers appear to be a natural part of the population. Salary distributions are naturally right-skewed. There are only 28 potential outliers among 8 122 observations (approximately 0.34% of the sample). We keep these observations because they are likely to represent genuine salaries rather than data errors.

The sample size is large (n = 8122). Therefore, by the Central Limit Theorem, the sampling distribution of the sample mean is approximately normal even if the population distribution is not.

### Secondly, let's state the null and alternative hypotheses. 

Null hypothesis $H_0: \mu_0 = 63500$

Alternative hypothesis $H_1​: μ_0 < 63500$

### Choosing the test

We want to test the population mean from one sample. Since the population variance is unknown, a one-sample t-test is appropriate.

The test statistic is 
$$
T = \frac{\bar X - \mu_0}{S/\sqrt{n}}
$$
 which follows Student's t-distribution with n−1 degrees of freedom under the null hypothesis.

In [64]:
salary = df_filtered["Зарплата"]

mu = salary.mean()
std = salary.std()
n = len(salary)
print(f"The data has mean = {round(mu, 2)}, std = {round(std, 2)}, n = {n}")

The data has mean = 62996.16, std = 25690.78, n = 8122


In [126]:
mu_0 = 64_200 - 700
task1_alpha = 0.05
critical_value = stats.t.ppf(task1_alpha, df=n-1)
print(f"Critical value is {round(critical_value, 2)}")

Critical value is -1.65


In [127]:
t = (mu - mu_0) / (std / math.sqrt(n))
print(f"The value of the test statistics is {round(t, 2)}")

The value of the test statistics is -1.77


Since the test statistic (-1.77) is less than the critical value (-1.645), we reject the null hypothesis at the 5% significance level. There is sufficient evidence that the mean salary in 2023 is lower than 63,500 rubles.

### Let's compute the p-value

In [67]:
p_value = stats.t.cdf(t, df=n-1)
print(f"The p-value is {round(p_value, 4)} -- the smallest significance level at which the null hypothesis would be rejected.")

The p-value is 0.0386 -- the smallest significance level at which the null hypothesis would be rejected.


### Fast-track workflow (Equivalent implementation using SciPy).

In [71]:
def hypothesis_testing(sample, popmean, alpha, alternanive):
    result = stats.ttest_1samp(
        salary,
        popmean=popmean,
        alternative=alternanive
    )

    print(f"t statistic = {result.statistic:.3f}")
    print(f"p-value = {result.pvalue:.4f}")

    if result.pvalue <= alpha:
        print(f"Since p-value <= {alpha}, reject H0.")
    else:
        print(f"Since p-value > {alpha}, fail to reject H0.")

In [78]:
salary_filtered = df.dropna()["Зарплата"]

mu0 = 63_500
popmean = mu0,
alternative = "less"
alpha = 0.05

hypothesis_testing(salary_filtered, mu0, alpha, alternative)

t statistic = -1.767
p-value = 0.0386
Since p-value <= 0.05, reject H0.


### What if we keep the missing values?

The salary column in the original dataset does not contain any missing values. Therefore, we can continue the analysis using the full set of observations.


In [79]:
salary = df["Зарплата"].dropna()

hypothesis_testing(salary, mu0, alpha, alternative)

t statistic = -1.333
p-value = 0.0912
Since p-value > 0.05, fail to reject H0.


The result obtained is opposite: at the stated level of significance 0.05 we fail to reject the null hypothesis.

In [80]:
print("Original mean:", salary.mean())
print("After dropna:", salary_filtered.mean())

print("Original median:", salary.median())
print("After dropna:",  salary_filtered.median())

Original mean: 63157.0371
After dropna: 62996.158827874904
Original median: 62011.0
After dropna: 61778.0


# Task 2

You hypothesize that in large Russian cities, approximately 65% people have a higher education. You have a sample. Test your hypothesis about the proportion of people with a higher education against a right-tailed alternative with a Type I error 10%.

Note! Higher education degrees include "бакалавр", "магистр", "доктор наук".

### First, let's inspect the data

We assume that each employee is recorded only once and that tha sample was collected independently in random. Nans are already dropped from the previous task. We need to take a look at the "Регион" and "Уровень образования" columns what correspond to the location and the education information.

In [103]:
cities = (
    df[["Регион", "Уровень образования"]]
    .rename(
        columns={
            "Регион": "city",
            "Уровень образования": "education"
        }
    )).dropna()
cities.tail()

,city,education
9994,Москва,бакалавр
9995,Санкт-Петербург,бакалавр
9997,Москва,бакалавр
9998,Нижний Новгород,среднее общее
9999,Москва,среднее профессиональное


In [108]:
print("Describe education data:")
cities["education"].value_counts()

Describe education data:


education
бакалавр                    2842
среднее общее               2006
магистр                     1842
доктор наук                  970
среднее профессиональное     947
Name: count, dtype: int64

In [109]:
print("Describe location data:")
cities["city"].value_counts()

Describe location data:


city
Москва             2479
Санкт-Петербург    1615
Новосибирск         835
Нижний Новгород     822
Екатеринбург        816
Казань              811
Челябинск           806
Омск                423
Name: count, dtype: int64

We have collected data on the population of Russian biggiest (top 50) cities for the 2024.

In [110]:
population = pd.read_csv("Population_2024.csv")
population.head()

,rank,city,population
0,1,Москва,13274285
1,2,Санкт-Петербург,5652922
2,3,Новосибирск,1637266
3,4,Екатеринбург,1548187
4,5,Казань,1329825


In [112]:
print(f"Check that all cities from the Salary file are presented in the Population file with the same spelling: {all(cities["city"].isin(population["city"]))}")

Check that all cities from the Salary file are presented in the Population file with the same spelling: True


Let's calculate the sampling fraction for each city: the number of the observations divided by the city's population.

In [113]:
city_counts = cities.groupby("city").size().reset_index(name="sample_size")

In [114]:
coverage = city_counts.merge(population, on="city")
coverage["fraction_per_1000"] = coverage["sample_size"] / coverage["population"] * 1000
coverage.sort_values("fraction_per_1000", ascending=False)

,city,sample_size,rank,population,fraction_per_1000
3,Нижний Новгород,822,7,1198245,0.686003
7,Челябинск,806,8,1176770,0.684926
1,Казань,811,5,1329825,0.609855
0,Екатеринбург,816,4,1548187,0.527068
4,Новосибирск,835,3,1637266,0.509997
5,Омск,423,13,1101367,0.384068
6,Санкт-Петербург,1615,2,5652922,0.285693
2,Москва,2479,1,13274285,0.186752


In [138]:
coverage["sample_size"].sum()

np.int64(8607)

In [123]:
lowest = coverage.loc[coverage["fraction_per_1000"].idxmin()]
highest = coverage.loc[coverage["fraction_per_1000"].idxmax()]
print(f"Cities with extreme sampling fractions:\n----- lowest:\n{lowest}\n----- highest:\n{highest}")

print(f"\nResidents from the {highest["city"]} are represented in the dataset {highest["fraction_per_1000"] / lowest["fraction_per_1000"]:.3f} times more frequent than residents from the {lowest["city"]}")

Cities with extreme sampling fractions:
----- lowest:
city                   Москва
sample_size              2479
rank                        1
population           13274285
fraction_per_1000    0.186752
Name: 2, dtype: object
----- highest:
city                 Нижний Новгород
sample_size                      822
rank                               7
population                   1198245
fraction_per_1000           0.686003
Name: 3, dtype: object

Residents from the Нижний Новгород are represented in the dataset 3.673 times more frequent than residents from the Москва


The sampling rate varies substantially across the cities. We also observe that the sampling fraction decreases as the population increases. The only exception is Omsk, the smallest city in the dataset, which is also relatively underrepresented.

In [97]:
high_education_labels = {"бакалавр", "магистр", "доктор наук"}

In [140]:
high_education_fraction = (
    cities.assign(high_ed=cities["education"].isin(high_education_labels))
    .groupby("city")
    .agg(
        sample_size=("education", "size"),
        high_ed=("high_ed", "sum"),
        high_ed_fraction=("high_ed", "mean")
    )
    .reset_index()
)

high_education_fraction


,city,sample_size,high_ed,high_ed_fraction
0,Екатеринбург,816,558,0.683824
1,Казань,811,542,0.668311
2,Москва,2479,1620,0.653489
3,Нижний Новгород,822,528,0.642336
4,Новосибирск,835,529,0.633533
5,Омск,423,286,0.676123
6,Санкт-Петербург,1615,1060,0.656347
7,Челябинск,806,531,0.658809


### Second, let's state the null and alternative hypothesis.

For each city we perform the hypothesis testing.

Null hypothesis $H_0$: $p_0 = 0.65$

Alternative hypothesis $H_1$: $p_0 > 0.65$

It asked to perform the rigth-tailed one-sample test for a population proportion with the 0.1 level of significance.

### Third, choosing the test

We need to check the population proportion of the respondents with higher education levels. 

The test statistic is 
$$
z = \frac{\hat p - p_0}{\sqrt{\frac{p_0 (1 - p_0)}{n}}}
$$
which follows standard normal distribution under the null hypothesis.

In [128]:
p_null = 0.65
task2_alpha = 0.1

In [168]:
task2_critical_value = stats.norm.ppf(1 - task2_alpha)
high_education_fraction["z_value"] = (high_education_fraction["high_ed_fraction"] - p_null) / np.sqrt(p_null * (1 - p_null) / high_education_fraction["sample_size"])

high_education_fraction["reject_null"] = high_education_fraction["z_value"] > task2_critical_value

high_education_fraction["p_value"] = stats.norm.sf(high_education_fraction["z_value"])

print(f"Critical value is {task2_critical_value:.3f}")
high_education_fraction


Critical value is 1.282


,city,sample_size,high_ed,high_ed_fraction,z_value,reject_null,p_value
0,Екатеринбург,816,558,0.683824,2.025691,True,0.021398
1,Казань,811,542,0.668311,1.093265,False,0.137139
2,Москва,2479,1620,0.653489,0.364240,False,0.357840
3,Нижний Новгород,822,528,0.642336,-0.460695,False,0.677491
4,Новосибирск,835,529,0.633533,-0.997628,False,0.840770
5,Омск,423,286,0.676123,1.126423,False,0.129993
6,Санкт-Петербург,1615,1060,0.656347,0.534745,False,0.296413
7,Челябинск,806,531,0.658809,0.524324,False,0.300026


### What if we do not separate the dataset by city?

In [169]:
total = high_education_fraction["sample_size"].sum()
high = high_education_fraction["high_ed"].sum()

fraction = high / total
print(f"Total fraction of the highly educated responders is {fraction:.3f}")

z_value = (fraction - p_null) / np.sqrt(p_null * (1 - p_null) / high_education_fraction["sample_size"].sum())
p_value = stats.norm.sf(z_value)

if z_value > task2_critical_value:
    print(f"Since z_value = {z_value:.3f} > {critical_value:.3f} we should reject H0")
else:
    print(f"Since z_value = {z_value:.3f} <= {critical_value:.3f} we fail to reject H0")

print(f"p-value is equal to {p_value:.3f}")

Total fraction of the highly educated responders is 0.657
Since z_value = 1.343 > 1.282 we should reject H0
p-value is equal to 0.090


# Task 3

It is hypothesized that the standard deviation of salaries in major cities is 25 300 RUB. You have a sample of salaries in major cities. Test this hypothesis against a two-sided alternative with a confidence level of 98%.

We are going to use salary DataFrame from the first task.

In [159]:
salary.info()

<class 'pandas.Series'>
RangeIndex: 10000 entries, 0 to 9999
Series name: Зарплата
Non-Null Count  Dtype  
--------------  -----  
10000 non-null  float64
dtypes: float64(1)
memory usage: 78.3 KB


### State the hypotheses

Null hypothesis $H_0$: $\sigma = 25\,300$

Alternative hypothesis $H_1$: $\sigma \neq 25\,300$

### Choosing the test

The test statistic is 
$$
\chi^2 = \frac{(n-1) s^2}{\sigma_0^2}
$$
that follows chi-squared distribution with n-1 degrees of freedom, where $\sigma_0 = 25\,300$. The chi-square test for a population variance assumes that the salary distribution is approximately normal.


In [177]:
task3_alpha = 0.02
n = salary.size
sigma_0 = 25_300
chi_squared = (n - 1) *(salary.std() ** 2) / sigma_0**2
chi_squared

np.float64(10336.409309180328)

In [173]:
critical_interval = (stats.chi2.ppf(task3_alpha / 2, df=n - 1), stats.chi2.ppf(1 - task3_alpha / 2, df=n - 1))
critical_interval

(np.float64(9672.965289892134), np.float64(10330.917127604109))

In [180]:
if chi_squared < critical_interval[0] or chi_squared > critical_interval[1]:
    print(f"The value of the test statistic {chi_squared:.3f} falls in the critical region.\nTherefore, we reject the null hypothesis.")
else:
    print(f"The value of the test statistic {chi_squared:.3f} does not fall in the critical region.\nTherefore, we fail to reject the null hypothesis.")

The value of the test statistic 10336.409 falls in the critical region.
Therefore, we reject the null hypothesis.


### Computing p-value

In [181]:
p_value = 2 * min(stats.chi2.cdf(chi_squared, df=n - 1), stats.chi2.sf(chi_squred, df=n - 1))
p_value

np.float64(0.018062070340782495)

# Task 4 

Using a confidence interval with a significance level of 0.01 for the proportion of people with the vocational education, test the hypothesis that this proportion is equal to 15%.

### Let's state the hypotheses

Null hypothesis $H_0$: $p = 0.15$

Alternative hypothesis $H_1$: $p \neq 0.15$

The test statistic is 
$$
\hat P = \frac{\hat p - p_0}{\sqrt{\frac{\hat p (1 - \hat p)}{n}}}
$$
which follows standard normal distribution under the null hypotesis, $p_0 = 0.15$ and $n$ is the sample size.

The confidence interval is constucted as follows
$$
(\hat p - z_{1 - \frac{\alpha}{2}} \sqrt{\frac{\hat p (1 - \hat p)}{n}}, \hat p + z_{1 - \frac{\alpha}{2}} \sqrt{\frac{\hat p (1 - \hat p)}{n}})
$$

### Second, get ready with the data


In [35]:
education = df["Уровень образования"].dropna().copy()

vocational_count = (education == "среднее профессиональное").sum()
p_hat = vocational_count / education.size
print(f"The value of the test statistic is {p_hat:.3f}")

The value of the test statistic is 0.110


In [32]:
task4_alpha = 0.01
p_0 = 0.15

z_value = stats.norm.ppf(1 - task4_alpha / 2)
delta = z_value * math.sqrt(p_0 * (1 - p_0) / education.size)

confidence_interval = (p_hat - delta, p_hat + delta)

In [34]:
if p_0 < confidence_interval[0] or p_0 > confidence_interval[1]:
    print(f"The p_0={p_0} is out of the confidence interval ({confidence_interval[0]:.3f}, {confidence_interval[1]:.3f})")
else:
    print(f"The p_0={p_0} falls in the confidence interval ({confidence_interval[0]:.3f}, {confidence_interval[1]:.3f})")

The p_0=0.15 is out of the confidence interval (0.101, 0.120)
